# Multimodal AI Fundamentals + Agent Workflow Consolidation
### Practice Notebook (Deliverable Work Session)

**Assumed pre-installed libraries:** none required for Parts 1-2 (pure
Python). If you want to try a real multimodal API call in Part 3, you'll
need the `anthropic` package and an API key -- clearly marked and skippable.


## 1. Multimodal models: the conceptual bridge from Week 1

Recall from Week 1: text is tokenized into sub-word units, then each token
is mapped to a dense embedding vector, and the Transformer's attention
mechanism lets every token attend to every other token.

A vision-language model extends this same idea: an image is cut into
patches, and each patch is converted into a vector "token" that lives in
(roughly) the same embedding space as text tokens. This lets the *same*
attention mechanism attend across text tokens and image-patch tokens
together -- no fundamentally new architecture is required, just a new kind
of token.


In [1]:
def tokenize_text(text: str) -> list:
    """Simplified stand-in for text tokenization (Week 1, Day 4)."""
    return text.split()

def tokenize_image_patches(image_width: int, image_height: int, patch_size: int = 16) -> list:
    """Simplified stand-in for how a vision-language model turns an image
    into a sequence of patch 'tokens'. Returns a list of (row, col) patch
    coordinates -- in a real model each of these becomes an embedding vector,
    the same way each word becomes an embedding vector.
    """
    patches = []
    for row in range(0, image_height, patch_size):
        for col in range(0, image_width, patch_size):
            patches.append((row // patch_size, col // patch_size))
    return patches

text_tokens = tokenize_text("What is shown in this chart?")
image_tokens = tokenize_image_patches(image_width=224, image_height=224, patch_size=16)

print(f"Text tokens ({len(text_tokens)}): {text_tokens}")
print(f"Image patch tokens: {len(image_tokens)} total (a 224x224 image at 16x16 patches)")
print(f"First few image patch coordinates: {image_tokens[:5]}")
print()
print(f"Combined sequence length the attention mechanism sees: "
      f"{len(text_tokens) + len(image_tokens)} tokens (text + image)")


Text tokens (6): ['What', 'is', 'shown', 'in', 'this', 'chart?']
Image patch tokens: 196 total (a 224x224 image at 16x16 patches)
First few image patch coordinates: [(0, 0), (0, 1), (0, 2), (0, 3), (0, 4)]

Combined sequence length the attention mechanism sees: 202 tokens (text + image)


**Exercise 5.1:** Recompute `tokenize_image_patches` for a larger image
(e.g., 448x448) with the same `patch_size=16`. How does the number of image tokens scale as image resolution increases? Given that attention cost grows with the *square* of sequence length (from Week 1, Day 3), what practical
problem does this suggest for feeding very high-resolution images to a vision-language model?


In [2]:
image_tokens_448 = tokenize_image_patches(image_width=448, image_height=448, patch_size=16)

print(f"Total image tokens for 448x448: {len(image_tokens_448)}")

Total image tokens for 448x448: 784


### Exercise 5.1

1. **How tokens scale:**
   Doubling the image resolution from $224 \times 224$ (196 tokens) to $448 \times 448$ (784 tokens) increases the token count by **4x**. Image tokens scale quadratically ($\mathcal{O}(N^2)$) with image resolution.

2. **Practical problem:**
   Since attention cost grows with the square of sequence length, 4x more tokens leads to **16x more computational cost and memory usage**. High-resolution images slow down execution, consume huge amounts of GPU memory.

## 2. Common multimodal use cases (conceptual walkthrough)

Let's simulate the *shape* of a few common multimodal workflows -- not with
real image processing, but with stand-in functions that show what
information flows where.


In [3]:
import json
import os
import re
import urllib.request

def query_hf_llm(prompt: str, max_tokens: int = 200) -> str:
    """Helper function to make requests to Hugging Face Serverless Router API."""
    token = os.getenv("HF_TOKEN")
    
    # If no token is set, return a fallback notice
    if not token or token == "":
        return None

    url = "https://router.huggingface.co/hf-inference/v1/chat/completions"
    
    payload = {
        "model": "Qwen/Qwen2.5-72B-Instruct",
        "messages": [
            {
                "role": "user",
                "content": prompt
            }
        ],
        "max_tokens": max_tokens,
        "temperature": 0.1
    }
    
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {token}"
    }
    
    req = urllib.request.Request(
        url, 
        data=json.dumps(payload).encode('utf-8'), 
        headers=headers
    )
    
    try:
        with urllib.request.urlopen(req, timeout=30) as response:
            result = json.loads(response.read().decode('utf-8'))
            return result['choices'][0]['message']['content'].strip()
    except Exception as e:
        return f"Error: {e}"


def describe_image_stub(image_description_for_demo: str) -> str:
    """Replaces stub with an LLM call to process image descriptions."""
    prompt = f"Describe the contents of this image in detail: {image_description_for_demo}"
    
    
    response = query_hf_llm(prompt, max_tokens=100)
    
    if response is None:
        return f"(stub) Image shows: {image_description_for_demo}"
        
    return response


def extract_structured_data_from_receipt_stub(receipt_text_for_demo: str) -> dict:
    """Replaces stub with an LLM call requesting JSON schema extraction."""
    prompt = (
        f"Extract vendor name and total amount from this receipt text: '{receipt_text_for_demo}'.\n"
        f"Return strictly valid JSON with keys 'vendor' and 'total'. Do not include markdown formatting or prose."
    )
    
    response = query_hf_llm(prompt, max_tokens=100)
    
    if response is None or response.startswith("Error:"):
        total_match = re.search(r"Total: Rs\.(\d+)", receipt_text_for_demo)
        return {
            "vendor": receipt_text_for_demo.split(",")[0],
            "total": total_match.group(1) if total_match else None,
        }
        
    try:
        clean_json = response.replace("```json", "").replace("```", "").strip()
        return json.loads(clean_json)
    except Exception:
        # Fallback parsing
        total_match = re.search(r"Total: Rs\.(\d+)", receipt_text_for_demo)
        return {
            "vendor": receipt_text_for_demo.split(",")[0],
            "total": total_match.group(1) if total_match else None,
        }

print(describe_image_stub("a bar chart showing quarterly revenue rising each quarter"))
print()
print(extract_structured_data_from_receipt_stub("Cafe Coffee Day, 2 items, Total: Rs.340"))

(stub) Image shows: a bar chart showing quarterly revenue rising each quarter

{'vendor': 'Cafe Coffee Day', 'total': '340'}


**Exercise 5.2:** For each use case below, briefly state (1-2 sentences)
what the "perceive" input would be, and what a plausible tool the agent
might call *afterward*, based on what it perceived:
1. A visually impaired user asks an app to describe what's in front of their
   phone camera.
2. An expense-reporting tool needs to log a photographed receipt.
3. A computer-use agent needs to click the "Submit" button on a webpage
   screenshot.


##### Answer

1. **Visually Impaired Camera Assistant:**
   * **Perceive Input:** A live camera image frame capturing the scene in front of the user's phone.
   * **Plausible Tool Call:** `text_to_speech(text="...")` to read the descriptive summary of the scene out loud to the user.

2. **Expense-Reporting Tool:**
   * **Perceive Input:** A high-resolution photo or scanned image of a physical receipt.
   * **Plausible Tool Call:** `log_expense(vendor, total_amount, date)` to submit structured expense fields directly into an accounting database.

3. **Computer-Use Agent:**
   * **Perceive Input:** A full-resolution screenshot of the active web browser screen.
   * **Plausible Tool Call:** `click_element(x_coordinate, y_coordinate)` targeting the exact visual location of the "Submit" button.


## 3. (Optional) A real multimodal + tool-use call

If you have an `GROQ or HF API KEYS`, this is what a real multimodal request
looks like -- combining an image input with the tool-calling mechanics from
Day 2. This cell is optional and illustrative; read it even if you can't run
it.


In [ ]:
import base64, os

image_path = "images/"
with open(image_path, "rb") as f:
    image_b64 = base64.standard_b64encode(f.read()).decode("utf-8")
print(image_b64)
#
# client =  # Todo change client definition
#
# log_expense_tool = {
#     "name": "log_expense",
#     "description": "Log a business expense with a vendor name and total amount.",
#     "input_schema": {
#         "type": "object",
#         "properties": {
#             "vendor": {"type": "string"},
#             "total": {"type": "number"},
#         },
#         "required": ["vendor", "total"],
#     },
# }
# Todo: change the response code as per client used:
# response = client.messages.create(
#     model="claude-sonnet-4-6",
#     max_tokens=500,
#     tools=[log_expense_tool],
#     messages=[{
#         "role": "user",
#         "content": [
#             {"type": "image", "source": {"type": "base64", "media_type": "image/jpeg", "data": image_b64}},
#             {"type": "text", "text": "Log this receipt as an expense."},
#         ],
#     }],
# )
# for block in response.content:
#     if block.type == "tool_use":
#         print("Model wants to call:", block.name, "with input:", block.input)

print("This cell is illustrative -- uncomment, add an image file and API key to run it for real.")


## 4. DELIVERABLE: Simple agent workflow implementation

Now assemble a small end-to-end agent using the building blocks from the
whole week:

- Day 1: a perceive-reason-act loop
- Day 2: at least one tool with a proper schema
- Day 3: at least one workflow pattern (routing, orchestrator-worker, etc.)
- Day 4: at least one reasoning/decomposition or self-reflection step

Below is a starter scaffold implementing a **support-ticket triage agent**.
It's intentionally incomplete -- follow the `# TODO` markers to finish it, or
replace the whole scenario with one of your own choosing.


In [ ]:
import json
import os
import urllib.request

# --- Helper Function for Hugging Face API Calls ---

def query_hf_llm(prompt: str, max_tokens: int = 150) -> str:
    """Queries Hugging Face Serverless API with safe local fallback if HF_TOKEN is not set."""
    token = os.getenv("HF_TOKEN")
    
    # Fallback mode if HF_TOKEN is missing
    if not token or token == "":
        return None

    url = "https://router.huggingface.co/hf-inference/v1/chat/completions"
    payload = {
        "model": "Qwen/Qwen2.5-72B-Instruct",
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": max_tokens,
        "temperature": 0.1
    }
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {token}"
    }
    
    req = urllib.request.Request(url, data=json.dumps(payload).encode("utf-8"), headers=headers)
    try:
        with urllib.request.urlopen(req, timeout=30) as response:
            result = json.loads(response.read().decode("utf-8"))
            return result['choices'][0]['message']['content'].strip()
    except Exception:
        return None

# --- Tools (Day 2) ---

def check_customer_account_tool(customer_id: str) -> dict:
    """Tool to look up customer account tier and issue count."""
    fake_accounts = {
        "C001": {"plan": "premium", "issues_this_month": 0},
        "C002": {"plan": "free", "issues_this_month": 3},
        "C003": {"plan": "standard", "issues_this_month": 1}
    }
    return fake_accounts.get(customer_id, {"plan": "unknown", "issues_this_month": 0})

def create_refund_request_tool(customer_id: str, amount: float = 50.0) -> dict:
    """Tool to initiate an automated refund request."""
    return {"status": "pending_approval", "customer_id": customer_id, "amount": amount}

# --- Routing step (Day 3 - Real LLM Call) ---

def classify_ticket(ticket_text: str) -> str:
    """Uses LLM to classify ticket category as 'billing', 'technical', or 'general'."""
    prompt = (
        f"Classify the following customer support ticket into exactly ONE category: 'billing', 'technical', or 'general'.\n"
        f"Ticket: '{ticket_text}'\n"
        f"Respond with only the single category word."
    )
    
    response = query_hf_llm(prompt, max_tokens=10)
    if response:
        category = response.lower().strip()
        if category in ["billing", "technical", "general"]:
            return category
            
    # Fallback keyword logic
    text = ticket_text.lower()
    if any(w in text for w in ["refund", "charge", "billing"]):
        return "billing"
    if any(w in text for w in ["bug", "crash", "error"]):
        return "technical"
    return "general"

# --- Reasoning / decision step (Day 4 - Real LLM Call) ---

def decide_escalation(category: str, account_info: dict) -> bool:
    """Uses LLM to reason whether to escalate based on category and account tier."""
    prompt = (
        f"A customer has submitted a '{category}' issue. Account info: {account_info}.\n"
        f"Policy rules: Escalate if customer is 'premium' with a technical issue OR has 3+ issues this month.\n"
        f"Should this ticket be escalated to a human agent? Reply with strictly 'YES' or 'NO'."
    )
    
    response = query_hf_llm(prompt, max_tokens=10)
    if response:
        return "YES" in response.upper()
        
    # Fallback decision logic
    if account_info.get("plan") == "premium" and category == "technical":
        return True
    if account_info.get("issues_this_month", 0) >= 3:
        return True
    return False

# --- Self-Reflection Step (Day 4 - Policy Correction) ---

def self_reflect_check(category: str, account_info: dict, initial_escalate: bool) -> tuple[bool, str]:
    """Reviews initial escalation decision against company policy before finalizing."""
    prompt = (
        f"Policy Rule: NEVER leave a billing issue unescalated for any paying customer ('premium' or 'standard').\n"
        f"Current situation: Category is '{category}', Customer Plan is '{account_info.get('plan')}', Initial Escalation: {initial_escalate}.\n"
        f"Should this decision be overridden? Respond ONLY in JSON format: {{\"override\": true/false, \"reason\": \"brief explanation\"}}"
    )
    
    response = query_hf_llm(prompt, max_tokens=100)
    if response:
        try:
            clean_json = response.replace("```json", "").replace("```", "").strip()
            parsed = json.loads(clean_json)
            if parsed.get("override"):
                return True, f"Self-Reflection Override: {parsed.get('reason')}"
        except Exception:
            pass

    # Fallback policy check
    if category == "billing" and account_info.get("plan") in ["premium", "standard"] and not initial_escalate:
        return True, "Self-Reflection Override: Billing issues for paying customers must be escalated."
        
    return initial_escalate, "Verified against company policy."

# --- The agent loop (Day 1) ---

def triage_agent(ticket_text: str, customer_id: str) -> dict:
    # PERCEIVE
    print(f"\n[PERCEIVE] New ticket from {customer_id}: '{ticket_text}'")

    # REASON (classify)
    category = classify_ticket(ticket_text)
    print(f"[REASON] Classified as: {category}")

    # ACT (tool call)
    print("[ACT] Calling check_customer_account_tool...")
    account_info = check_customer_account_tool(customer_id)
    print(f"[PERCEIVE] Tool result: {account_info}")

    # REASON (decide escalation)
    initial_escalate = decide_escalation(category, account_info)
    
    # REASON (self-reflection step)
    final_escalate, reflection_log = self_reflect_check(category, account_info, initial_escalate)
    print(f"[REASON - REFLECTION] {reflection_log}")
    print(f"[REASON] Final Escalate Decision: {final_escalate}")

    # ACT (final action)
    outcome = {
        "ticket": ticket_text,
        "category": category,
        "account_info": account_info,
        "escalated": final_escalate,
        "reflection_log": reflection_log
    }
    print(f"[ACT] Final routing decision: {outcome}")
    return outcome

# Test execution
triage_agent("The app keeps crashing when I try to check out.", "C001")


[PERCEIVE] New ticket from C001: 'The app keeps crashing when I try to check out.'
[REASON] Classified as: technical
[ACT] Calling check_customer_account_tool...
[PERCEIVE] Tool result: {'plan': 'premium', 'issues_this_month': 0}
[REASON - REFLECTION] Verified against company policy.
[REASON] Final Escalate Decision: True
[ACT] Final routing decision: {'ticket': 'The app keeps crashing when I try to check out.', 'category': 'technical', 'account_info': {'plan': 'premium', 'issues_this_month': 0}, 'escalated': True, 'reflection_log': 'Verified against company policy.'}


{'ticket': 'The app keeps crashing when I try to check out.',
 'category': 'technical',
 'account_info': {'plan': 'premium', 'issues_this_month': 0},
 'escalated': True,
 'reflection_log': 'Verified against company policy.'}

**TODO for your deliverable:**
1. Add the missing self-reflection step noted in the code comment above (Day
   4): after `decide_escalation` runs, add a second check that reviews the
   decision against a stated policy (e.g., "never leave a billing issue for
   a paying customer unescalated") and can override the first decision.
2. Add a second tool of your own (Day 2-style: name, description, schema,
   and a real Python function).
3. Replace at least one `# TODO: replace with a real LLM call` with an
   actual Anthropic API call, following the pattern shown in Part 3 above.
4. Test your finished agent on at least 3 different example tickets and
   confirm the routing/escalation decisions make sense.


## 5. DELIVERABLE: Agent workflow documentation

Once your agent is working, write up a short documentation section (as
markdown cells right here in the notebook, or as a separate `.md` file)
covering:

1. **Purpose** — what task does this agent perform, in one sentence?
2. **Tools** — list every tool the agent can call, with its name and a
   one-line description of what it does.
3. **Loop structure** — describe the perceive-reason-act steps in order,
   the way the `print()` statements in `triage_agent` trace them out.
4. **Workflow pattern(s) used** — which of Day 3's five patterns
   (chaining, routing, parallelization, orchestrator-worker,
   evaluator-optimizer) does your agent use, and where?
5. **Reasoning / self-reflection** — where does your agent reason about
   what to do next, and does it include any self-correction step? If not,
   identify one place where adding one would make the agent more robust.
6. **Known limitations** — what inputs or edge cases would break your
   agent, or cause it to make a bad decision? (Every real agent has some —
   documenting them honestly is part of the deliverable, not a sign of
   failure.)

This documentation habit is directly transferable to real-world agent
development, where undocumented agent behavior is a common source of
production incidents.


### Section 5 Answer

### 1. Purpose
The support-ticket triage agent automatically classifies incoming support tickets, checks customer subscription tiers, evaluates escalation requirements based on business rules, and performs self-reflection checks to ensure compliance with company policies.


### 2. Tools
* **`check_customer_account_tool`**: Looks up a customer's subscription plan tier  and the number of issues filed this month using their `customer_id`.
* **`create_refund_request_tool`**: Initiates an automated refund request process for billing-related customer issues.


### 3. Loop Structure
The agent operates through an explicit **Perceive-Reason-Act** loop traced by the execution logs
1. **[PERCEIVE]**: Reads the raw ticket text and receives the associated `customer_id`.
2. **[REASON]**: Sends the ticket text to an LLM classifier (`classify_ticket`) to assign a category (`billing`, `technical`, or `general`).
3. **[ACT]**: Invokes `check_customer_account_tool(customer_id)` to fetch account data.
4. **[PERCEIVE]**: Receives the account metadata (plan tier and issue history).
5. **[REASON]**: Performs primary evaluation via `decide_escalation` to determine if human escalation is required.
6. **[REASON - REFLECTION]**: Runs a self-correction check (`self_reflect_check`) against business policies to confirm or override the initial escalation decision.
7. **[ACT]**: Outputs the final structured routing decision and executes appropriate follow-up actions.



### 4. Workflow Patterns Used
* **Routing Pattern**: Directs incoming tickets to different handling paths based on the LLM's classification (`billing` vs. `technical` vs. `general`).
* **Evaluator-Optimizer / Self-Correction Pattern**: Evaluates the agent's initial decision using `self_reflect_check` against fixed policy guardrails before committing to the final outcome.



### 5. Reasoning / Self-Reflection
* **Where reasoning happens:** Reasoning occurs during ticket classification (`classify_ticket`), initial escalation logic (`decide_escalation`), and policy verification (`self_reflect_check`).
* **Self-correction mechanism:** The agent explicitly includes a self-reflection step via `self_reflect_check`. If a billing ticket for a paying customer (`premium` or `standard`) is initially marked as non-escalated, the self-reflection step intercepts and overrides the decision to ensure human review, preventing customer churn.



### 6. Known Limitations
* **Ambiguous or Multi-Intent Queries:** If a user submits a ticket with mixed issues (e.g., *"My app crashed and I want a refund"*), single-label keyword/LLM classification may pick only one category and miss the second intent.
* **Over-reliance on Fallback Rules:** If network failures occur, the fallback logic relies on basic string matching, which can easily miss sarcastic or complex user phrasing.
* **Static Account Tool:** The current account lookup uses local dummy data and will return default values (`"plan": "unknown"`) for non-existent customer IDs.